imports and config

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"

TARGET_COL = "loan_paid_back"
ID_COL = "id"

load data

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)

display(train_df.head())
display(test_df.head())

define columns for preprocessing

In [ ]:
# columns to exclude for now
drop_columns = [
    "id",
    "employment_status",   # pending leakage review
]

# target encoding candidate
target_encode_columns = [
    "grade_subgrade",
]

# one-hot columns
one_hot_columns = [
    "loan_purpose",
    "education_level",
    "gender",
]

# keep only columns that actually exist
drop_columns = [col for col in drop_columns if col in train_df.columns]
target_encode_columns = [col for col in target_encode_columns if col in train_df.columns]
one_hot_columns = [col for col in one_hot_columns if col in train_df.columns]

# numeric columns = all numeric except target and dropped columns
numeric_columns = [
    col for col in train_df.select_dtypes(include=["number"]).columns
    if col not in [TARGET_COL] + drop_columns
]

print("drop_columns:", drop_columns)
print("target_encode_columns:", target_encode_columns)
print("one_hot_columns:", one_hot_columns)
print("numeric_columns:", numeric_columns)

split X and y

In [ ]:
X_train = train_df.drop(columns=[TARGET_COL] + drop_columns, errors="ignore")
y_train = train_df[TARGET_COL].copy()

X_test = test_df.drop(columns=drop_columns, errors="ignore")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)

build preprocessing pipelines

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

one_hot_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

target_encoding_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("target_encoder", TargetEncoder(
            smooth="auto",
            cv=5,
            shuffle=True,
            random_state=42
        )),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, [col for col in numeric_columns if col in X_train.columns]),
        ("ohe", one_hot_pipeline, [col for col in one_hot_columns if col in X_train.columns]),
        ("te", target_encoding_pipeline, [col for col in target_encode_columns if col in X_train.columns]),
    ],
    remainder="drop"
)

preprocessor

fit and transform train/test

In [ ]:
X_train_prepared = preprocessor.fit_transform(X_train, y_train)
X_test_prepared = preprocessor.transform(X_test)

print("Prepared train shape:", X_train_prepared.shape)
print("Prepared test shape:", X_test_prepared.shape)

get feature names

In [ ]:
feature_names = preprocessor.get_feature_names_out()

prepared_train_df = pd.DataFrame(
    X_train_prepared,
    columns=feature_names,
    index=X_train.index
)

prepared_test_df = pd.DataFrame(
    X_test_prepared,
    columns=feature_names,
    index=X_test.index
)

display(prepared_train_df.head())
print("Number of prepared features:", len(feature_names))